# Giáo trình Dữ liệu lớn – Chương 5

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume và thay `data/` bằng `/Volumes/<catalog>/<schema>/<volume>/`.

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch{ch:02d}/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
!pip install -q pyspark==3.5.7 pyarrow==16.1.0 pandas==2.2.2
import os
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch05").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 5.1. Thống kê mô tả với describe và summary.


In [ ]:
# Doc du lieu khach hang tu HDFS
df = spark.read.parquet("data/khach_hang")
print("So dong:", df.count(), "- So cot:", len(df.columns))
df.printSchema()

# Nam thong ke co ban cho cac cot so
df.describe("tuoi", "thu_nhap", "chi_tieu").show()

# summary cho phep chi dinh them cac phan vi
df.select("tuoi", "thu_nhap") \
  .summary("count", "mean", "stddev", "min",
           "25%", "50%", "75%", "max") \
  .show()

## Đoạn mã 5.2. Đếm số giá trị null trên từng cột trong một lần quét.


In [ ]:
from pyspark.sql import functions as F

# Dem so gia tri null cua moi cot (mot job duy nhat)
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show()

# Tinh ty le null theo cot de quyet dinh chien luoc xu ly
tong_so_dong = df.count()
ty_le_null = df.select([
    (F.count(F.when(F.col(c).isNull(), c)) / tong_so_dong).alias(c)
    for c in df.columns
])
ty_le_null.show()

## Đoạn mã 5.3. Phân phối tần suất, phân vị xấp xỉ và tương quan.


In [ ]:
# Phan phoi tan suat cua bien dinh danh
df.groupBy("thanh_pho").count() \
  .orderBy(F.desc("count")) \
  .show(10)

# Phan vi xap xi voi sai so tuong doi 0.01
q1, q2, q3 = df.approxQuantile("thu_nhap",
                               [0.25, 0.5, 0.75], 0.01)
print("Tu phan vi thu nhat:", q1)
print("Trung vi xap xi   :", q2)
print("Tu phan vi thu ba :", q3)

# He so tuong quan Pearson giua hai bien so
r = df.stat.corr("thu_nhap", "chi_tieu")
print("Tuong quan thu nhap - chi tieu:", r)

## Đoạn mã 5.4. Lấy mẫu và chuyển sang pandas để trực quan hóa.


In [ ]:
# Lay mau ngau nhien 1 phan tram, khong hoan lai, seed co dinh
mau = df.sample(withReplacement=False, fraction=0.01, seed=42)
print("Kich thuoc mau:", mau.count())  # kiem tra truoc khi keo ve

pdf = mau.select("tuoi", "thu_nhap", "chi_tieu").toPandas()

import matplotlib.pyplot as plt
pdf["thu_nhap"].hist(bins=50)
plt.xlabel("Thu nhap")
plt.ylabel("Tan suat")
plt.savefig("phan_phoi_thu_nhap.png")

## Đoạn mã 5.5. Phát hiện lệch phân phối và biến đổi logarit.


In [ ]:
# He so lech va do nhon cua bien thu nhap
df.select(F.skewness("thu_nhap").alias("do_lech"),
          F.kurtosis("thu_nhap").alias("do_nhon")).show()

# So sanh trung binh voi trung vi xap xi
tb = df.agg(F.mean("thu_nhap")).first()[0]
tv = df.approxQuantile("thu_nhap", [0.5], 0.01)[0]
print("Trung binh:", tb, "- Trung vi:", tv)

# Bien doi logarit lam giam do lech phai
df = df.withColumn("log_thu_nhap", F.log1p(F.col("thu_nhap")))

## Đoạn mã 5.6. Xóa dòng thiếu với dropna: how, thresh và subset.


In [ ]:
# Xoa dong co bat ky gia tri null nao (mac dinh how="any")
df_a = df.dropna(how="any")

# Chi xoa khi TAT CA cac cot deu null
df_b = df.dropna(how="all")

# Giu lai dong co it nhat 5 gia tri khac null
df_c = df.dropna(thresh=5)

# Chi xet null tren cac cot quan trong doi voi bai toan
df_d = df.dropna(subset=["thu_nhap", "tuoi"])

print("Truoc:", df.count(), "- Sau khi loc:", df_d.count())

## Đoạn mã 5.7. Điền khuyết bằng fillna và Imputer với trung vị.


In [ ]:
# Dien hang so an dinh rieng cho tung cot
df_dien = df.fillna({"thu_nhap": 0.0,
                     "so_don_hang": 0,
                     "thanh_pho": "khong_ro"})

# Imputer: hoc trung vi tren tap huan luyen roi ap dung lai
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=["tuoi", "thu_nhap"],
    outputCols=["tuoi_imp", "thu_nhap_imp"],
    strategy="median"          # hoac "mean", "mode"
)
model_imp = imputer.fit(df)    # hoc gia tri dien
df_imp = model_imp.transform(df)
model_imp.surrogateDF.show()   # xem gia tri da hoc

## Đoạn mã 5.8. Lọc ngoại lai theo quy tắc IQR với approxQuantile.


In [ ]:
# Tinh Q1, Q3 xap xi va suy ra hai nguong loc
q1, q3 = df.approxQuantile("thu_nhap", [0.25, 0.75], 0.01)
iqr = q3 - q1
can_duoi = q1 - 1.5 * iqr
can_tren = q3 + 1.5 * iqr

df_loc = df.filter(
    (F.col("thu_nhap") >= can_duoi) &
    (F.col("thu_nhap") <= can_tren)
)
so_ngoai_lai = df.count() - df_loc.count()
print("So quan sat bi loai:", so_ngoai_lai)

## Đoạn mã 5.9. Winsorize giá trị vượt ngưỡng bằng when/otherwise.


In [ ]:
df_wins = df.withColumn(
    "thu_nhap_w",
    F.when(F.col("thu_nhap") > can_tren, can_tren)
     .when(F.col("thu_nhap") < can_duoi, can_duoi)
     .otherwise(F.col("thu_nhap"))
)
df_wins.select("thu_nhap", "thu_nhap_w").describe().show()

## Đoạn mã 5.10. StringIndexer và IndexToString trên cột nghề nghiệp.


> Đoạn mã dùng `df_train` – ô chuẩn bị bên dưới tạo tập huấn luyện/kiểm tra từ `df_imp` (Đoạn mã 5.7).


In [ ]:
# Chuan bi: tap huan luyen/kiem tra tu du lieu da dien gia tri thieu (Doan ma 5.7)
df_train, df_test = df_imp.randomSplit([0.8, 0.2], seed=42)
print("train:", df_train.count(), "- test:", df_test.count())

In [ ]:
from pyspark.ml.feature import StringIndexer, IndexToString

indexer = StringIndexer(
    inputCol="nghe_nghiep",
    outputCol="nghe_nghiep_idx",
    handleInvalid="keep"   # nhan la duoc gom vao chi so cuoi
)
model_idx = indexer.fit(df_train)      # hoc bang tan suat
df_idx = model_idx.transform(df_train)

# Danh sach nhan theo thu tu tan suat giam dan
print(model_idx.labelsArray[0])
# vi du: ['nhan_vien', 'ky_su', 'giao_vien', 'bac_si', ...]

# Chuyen nguoc chi so ve nhan goc khi dien giai ket qua
inverter = IndexToString(
    inputCol="nghe_nghiep_idx",
    outputCol="nghe_nghiep_goc",
    labels=model_idx.labelsArray[0]
)
df_goc = inverter.transform(df_idx)

## Đoạn mã 5.11. OneHotEncoder cho hai cột nghề nghiệp và thành phố.


> Đoạn mã giả định `thanh_pho` đã được chỉ số hóa – ô chuẩn bị bên dưới bổ sung cột `thanh_pho_idx`.


In [ ]:
# Chuan bi: chi so hoa them cot thanh_pho de co thanh_pho_idx
from pyspark.ml.feature import StringIndexer
df_idx = (StringIndexer(inputCol="thanh_pho", outputCol="thanh_pho_idx", handleInvalid="keep")
          .fit(df_idx).transform(df_idx))

In [ ]:
from pyspark.ml.feature import OneHotEncoder

# Gia su thanh_pho da duoc StringIndexer hoa thanh thanh_pho_idx
encoder = OneHotEncoder(
    inputCols=["nghe_nghiep_idx", "thanh_pho_idx"],
    outputCols=["nghe_nghiep_vec", "thanh_pho_vec"],
    dropLast=True
)
model_ohe = encoder.fit(df_idx)
df_ohe = model_ohe.transform(df_idx)

df_ohe.select("nghe_nghiep", "nghe_nghiep_idx",
              "nghe_nghiep_vec").show(5, truncate=False)
# vi du ket qua: ky_su | 1.0 | (4,[1],[1.0])

## Đoạn mã 5.12. VectorAssembler gom các cột thành vector features.


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["tuoi", "thu_nhap", "chi_tieu",
               "nghe_nghiep_vec", "thanh_pho_vec"],
    outputCol="features_raw",
    handleInvalid="skip"
)
df_vec = assembler.transform(df_ohe)
df_vec.select("features_raw").show(3, truncate=False)

## Đoạn mã 5.13. Chuỗi assembler và scaler, fit trên train rồi transform test.


In [ ]:
from pyspark.ml.feature import StandardScaler

# Chia du lieu TRUOC khi hoc bat ky tham so nao
train, test = df_vec.randomSplit([0.8, 0.2], seed=42)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,   # tru trung binh (can vector day du)
    withStd=True     # chia do lech chuan
)
scaler_model = scaler.fit(train)        # hoc mu, sigma tren TRAIN
train_scaled = scaler_model.transform(train)
test_scaled = scaler_model.transform(test)  # ap dung nguyen ven

## Đoạn mã 5.14. MinMaxScaler và RobustScaler trên cùng cột vector.


In [ ]:
from pyspark.ml.feature import MinMaxScaler, RobustScaler

# Co gian tuyen tinh ve doan [0, 1]
mm = MinMaxScaler(inputCol="features_raw",
                  outputCol="features_mm")
train_mm = mm.fit(train).transform(train)

# Chuan hoa ben vung theo trung vi va khoang tu phan vi
rb = RobustScaler(inputCol="features_raw",
                  outputCol="features_rb",
                  withCentering=True, withScaling=True)
train_rb = rb.fit(train).transform(train)

## Đoạn mã 5.15. Pipeline hoàn chỉnh: StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler.


> Pipeline này giả định các cột số không còn giá trị thiếu – ô chuẩn bị bên dưới dùng kết quả Imputer (Đoạn mã 5.7).


In [ ]:
# Chuan bi: dung cac cot da dien gia tri thieu (5.7) lam dau vao cho Pipeline
df = (df_imp.drop("tuoi", "thu_nhap")
      .withColumnRenamed("tuoi_imp", "tuoi").withColumnRenamed("thu_nhap_imp", "thu_nhap")
      .dropna(subset=["chi_tieu"]))

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler)

idx_nghe = StringIndexer(inputCol="nghe_nghiep",
                         outputCol="nghe_idx",
                         handleInvalid="keep")
idx_tp = StringIndexer(inputCol="thanh_pho",
                       outputCol="tp_idx",
                       handleInvalid="keep")
ohe = OneHotEncoder(inputCols=["nghe_idx", "tp_idx"],
                    outputCols=["nghe_vec", "tp_vec"])
assembler = VectorAssembler(
    inputCols=["tuoi", "thu_nhap", "chi_tieu",
               "nghe_vec", "tp_vec"],
    outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw",
                        outputCol="features",
                        withMean=True, withStd=True)

pipeline = Pipeline(stages=[idx_nghe, idx_tp, ohe,
                            assembler, scaler])

train, test = df.randomSplit([0.8, 0.2], seed=42)
pipe_model = pipeline.fit(train)   # moi Estimator hoc tren train
train_ready = pipe_model.transform(train)
test_ready = pipe_model.transform(test)
train_ready.select("features").show(3, truncate=False)

## Đoạn mã 5.16. Lưu và nạp lại PipelineModel.


> Đoạn mã dùng `du_lieu_moi` (dữ liệu mới cùng lược đồ) – ô chuẩn bị bên dưới lấy tập `test` của Đoạn mã 5.15 làm ví dụ.


In [ ]:
# Chuan bi: du lieu moi cung luoc do voi du lieu huan luyen (lay tap test cua Doan ma 5.15)
du_lieu_moi = test

In [ ]:
# Luu pipeline da huan luyen xuong HDFS
duong_dan = "models/pipeline_tien_xu_ly"
pipe_model.write().overwrite().save(duong_dan)

# Nap lai o phien lam viec khac (moi truong trien khai)
from pyspark.ml import PipelineModel
pipe_model_2 = PipelineModel.load(duong_dan)
# du_lieu_moi: DataFrame moi, cung luoc do voi du lieu huan luyen
du_lieu_moi_ready = pipe_model_2.transform(du_lieu_moi)

## Đoạn mã 5.17. Khung winsorize cho cột thu nhập.


In [ ]:
q1, q3 = df.approxQuantile("thu_nhap", [0.25, 0.75], 0.01)
iqr = q3 - q1
can_duoi, can_tren = q1 - 1.5 * iqr, q3 + 1.5 * iqr
so_ngoai_lai = df.filter((F.col("thu_nhap") < can_duoi) |
                         (F.col("thu_nhap") > can_tren)).count()
df_w = df.withColumn(
    "thu_nhap_w",
    F.when(F.col("thu_nhap") > can_tren, can_tren)
     .when(F.col("thu_nhap") < can_duoi, can_duoi)
     .otherwise(F.col("thu_nhap")))

## Đoạn mã 5.18. Khung pipeline tiền xử lý cho bài toán churn.


> Khung lời giải Bài 5.2 trên `data/churn.csv` (cùng tập dữ liệu với Chương 6); PipelineModel được lưu vào `models/pipe_churn`.


In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (Imputer, StringIndexer,
    OneHotEncoder, VectorAssembler, StandardScaler)

df = spark.read.csv("data/churn.csv",
                    header=True, inferSchema=True)
train, test = df.randomSplit([0.8, 0.2], seed=42)

imp = Imputer(inputCols=["tuoi", "cuoc_hang_thang"],
              outputCols=["tuoi_imp", "cuoc_imp"],
              strategy="median")
idx = StringIndexer(inputCols=["goi_cuoc", "khu_vuc"],
                    outputCols=["goi_idx", "kv_idx"],
                    handleInvalid="keep")
ohe = OneHotEncoder(inputCols=["goi_idx", "kv_idx"],
                    outputCols=["goi_vec", "kv_vec"])
asm = VectorAssembler(
    inputCols=["tuoi_imp", "so_thang_su_dung", "cuoc_imp",
               "so_lan_goi_ho_tro", "goi_vec", "kv_vec"],
    outputCol="features_raw", handleInvalid="skip")
scl = StandardScaler(inputCol="features_raw", outputCol="features",
                     withMean=True, withStd=True)

pipe = Pipeline(stages=[imp, idx, ohe, asm, scl])
model = pipe.fit(train)              # chi hoc tren train
train_ready = model.transform(train)
test_ready = model.transform(test)
model.write().overwrite().save("models/pipe_churn")
model2 = PipelineModel.load("models/pipe_churn")

## Đoạn mã 5.19. Đoạn mã chuẩn hóa có một lỗi phương pháp nghiêm trọng.


> **Khung / minh họa – không chạy trực tiếp:** ví dụ về LỖI phương pháp (fit scaler trên toàn bộ dữ liệu) – không nên chạy theo.


In [ ]:
scaler_model = scaler.fit(df_vec)  # hoc tren TOAN BO du lieu
df_scaled = scaler_model.transform(df_vec)
train, test = df_scaled.randomSplit([0.8, 0.2], seed=42)